# **BizFlow360 LightGBM Model**
* **By:** Edusei Mikel
* **Date:** 7th August, 2026

**Imports, Data Loading, Preprocessing and Splitting**

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import lightgbm as lgb

# Load the synthetic data
df = pd.read_csv('../data/synthetic_msme_data.csv')

# Encode categorical variables
le_county = LabelEncoder()
le_sector = LabelEncoder()
df['county_encoded'] = le_county.fit_transform(df['county'])
df['sector_encoded'] = le_sector.fit_transform(df['sector'])

# Define Features (X) and Target (y)
features = [
    'business_age_months', 'employees', 
    'monthly_revenue_kes', 'monthly_expenses_kes', 
    'total_assets_kes', 'total_liabilities_kes', 
    'loan_amount_kes', 'mpesa_volume_kes', 
    'county_encoded', 'sector_encoded'
]

X = df[features]
y = df['distress_label']

# Split and Scale (Same split/scaler for fair comparison)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Data loaded, preprocessed, and split successfully.")

✅ Data loaded, preprocessed, and split successfully.


**Training the LightGBM Model**

In [3]:
print("Training LightGBM Model...")

# Initialize and train the model
# verbose=-1 suppresses training output to keep the notebook clean
lgb_model = lgb.LGBMClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    random_state=42, 
    verbose=-1
)

lgb_model.fit(X_train_scaled, y_train)

print("✅ LightGBM model trained successfully!")

Training LightGBM Model...
✅ LightGBM model trained successfully!


**Model Evaluation**

In [4]:
# Make predictions
y_pred = lgb_model.predict(X_test_scaled)
y_prob = lgb_model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("="*40)
print(" LIGHTGBM MODEL METRICS")
print("="*40)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("="*40)

 LIGHTGBM MODEL METRICS
Accuracy:  0.7470
Precision: 0.7426
Recall:    0.7560
F1-Score:  0.7493
ROC-AUC:   0.8322


**Saving the Model**

In [5]:
# Ensure trained models directory exists
os.makedirs('../models/trained/on_synthetic_data', exist_ok=True)

# Save the model
joblib.dump(lgb_model, '../models/trained/on_synthetic_data/lightgbm.joblib')

print("✅ LightGBM model saved to ml_models/models/trained/on_synthetic_data/lightgbm.joblib")

✅ LightGBM model saved to ml_models/models/trained/on_synthetic_data/lightgbm.joblib
